In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

week11_data = pd.read_csv('/content/sales_integrated.csv')

print("Dataset shape:", week11_data.shape)
week11_data.head()

Dataset shape: (130402, 55)


,so_id,line_number,product_id,quantity,unit_price,gst_rate,line_total,gst_amount,line_grand_total,customer_id,branch_id,order_date,delivery_date,order_status,payment_terms,total_order_value,total_gst_amount,grand_total,sales_channel,product_name,category,machine_type,brand,model_compatibility,unit_cost,unit_price_product,margin_percentage,gst_rate_product,weight_kg,dimensions_cm,material_type,warranty_months,reorder_level,safety_stock,max_stock_level,lead_time_days,criticality_level,usage_frequency,uom,last_purchase_price,last_purchase_date,customer_type,industry_segment,city,state,pincode,region,branch_id_customer,credit_limit,current_balance,payment_terms_customer,customer_since,last_purchase_date_customer,total_purchase_value,customer_rating
0,SO-990591,1,P018,19,11300,28,214700,60116.0,274816.0,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale,Wheel Rim WR-24,Undercarriage,Bulldozer,CAT,CAT D6R,8200,11300,37.8,28,24.00,70x25x25,steel,12,12,6,30,20,High,Medium,piece,8000,2024-02-20,Government,Logistics,Ahmedabad,Gujarat,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
1,SO-990591,2,P026,2,310,18,620,111.6,731.6,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale,Grease Cartridge GC-5,Lubricants,All,Generic,All Models,180,310,72.2,18,0.45,25x6x6,synthetic,3,200,100,500,4,Low,High,cartridge,170,2020-04-02,Government,Logistics,Ahmedabad,Gujarat,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
2,SO-990591,3,P026,3,310,18,930,167.4,1097.4,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale,Grease Cartridge GC-5,Lubricants,All,Generic,All Models,180,310,72.2,18,0.45,25x6x6,synthetic,3,200,100,500,4,Low,High,cartridge,170,2020-04-02,Government,Logistics,Ahmedabad,Gujarat,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
3,SO-990591,4,P013,9,1450,18,13050,2349.0,15399.0,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale,Seal Kit SK-15,Seals,Excavator,CAT,CAT 320D,950,1450,52.6,18,0.60,15x10x6,rubber,6,80,40,200,9,Medium,High,set,920,2022-12-19,Government,Logistics,Ahmedabad,Gujarat,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3
4,SO-990591,5,P004,3,39800,28,119400,33432.0,152832.0,C0070,AHM001,2022-05-24,2022-05-28,Delivered,Net 30,1345720,374439.6,1720159.6,Counter Sale,Track Chain TC-45,Undercarriage,Bulldozer,Volvo,Volvo D6,32500,39800,22.5,28,210.00,180x40x35,steel,18,8,4,20,30,High,Low,set,32000,2020-06-18,Government,Logistics,Ahmedabad,Gujarat,784913,West,AHM001,549454,84153,Net 30,2022-11-10,2024-12-04,4644233,3


In [2]:
week11_data["order_date"] = pd.to_datetime(
    week11_data["order_date"],
    errors="coerce"
)

numeric_columns = [
    "quantity",
    "line_total",
    "unit_price",
    "unit_cost",
    "margin_percentage",
    "reorder_level",
    "safety_stock",
    "max_stock_level",
    "lead_time_days",
    "customer_rating"
]

for col in numeric_columns:
    if col in week11_data.columns:
        week11_data[col] = pd.to_numeric(
            week11_data[col],
            errors="coerce"
        )

### **Data Quality Check**

In [3]:
print("Rows:", len(week11_data))
print("Columns:", len(week11_data.columns))

print(
    "Duplicate rows:",
    week11_data.duplicated().sum()
)

missing = (
    week11_data.isna()
    .sum()
    .sort_values(ascending=False)
)

missing[missing > 0].head(20)

Rows: 130402
Columns: 55
Duplicate rows: 0


,0


### **Executive KPI Dataset**

In [4]:
total_revenue = week11_data["line_total"].sum()
total_orders = week11_data["so_id"].nunique()
total_units = week11_data["quantity"].sum()
total_products = week11_data["product_id"].nunique()
total_customers = week11_data["customer_id"].nunique()
total_branches = week11_data["branch_id"].nunique()

average_order_value = (
    total_revenue / total_orders
)

executive_kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Total Orders",
        "Total Units Sold",
        "Total Products",
        "Total Customers",
        "Total Branches",
        "Average Order Value"
    ],
    "Value": [
        total_revenue,
        total_orders,
        total_units,
        total_products,
        total_customers,
        total_branches,
        average_order_value
    ]
})

executive_kpis

,KPI,Value
0,Total Revenue,2.544739e+10
1,Total Orders,2.000000e+04
2,Total Units Sold,1.368534e+06
3,Total Products,3.000000e+01
4,Total Customers,5.000000e+02
5,Total Branches,6.000000e+00
6,Average Order Value,1.272370e+06


### **Monthly Sales Analysis**

In [5]:
monthly_sales = (
    week11_data
    .groupby(
        pd.Grouper(
            key="order_date",
            freq="ME"
        )
    )
    .agg(
        revenue=("line_total", "sum"),
        units=("quantity", "sum"),
        orders=("so_id", "nunique"),
        customers=("customer_id", "nunique")
    )
    .reset_index()
)

monthly_sales["revenue_growth"] = (
    monthly_sales["revenue"]
    .pct_change() * 100
)

monthly_sales.head()

,order_date,revenue,units,orders,customers,revenue_growth
0,2019-01-31,392951210,19023,286,215,NaN
1,2019-02-28,281974060,15213,237,185,-28.241967
2,2019-03-31,368326180,19796,294,228,30.624136
3,2019-04-30,368885680,19672,281,219,0.151903
4,2019-05-31,364324690,20130,285,215,-1.236424


### **Product Performance**

In [6]:
product_performance = (
    week11_data
    .groupby(
        ["product_id", "product_name"],
        as_index=False
    )
    .agg(
        total_revenue=("line_total", "sum"),
        total_units=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        customers_served=("customer_id", "nunique"),
        branches_served=("branch_id", "nunique")
    )
)

product_performance["average_revenue_per_order"] = (
    product_performance["total_revenue"]
    / product_performance["total_orders"]
)

product_performance = (
    product_performance
    .sort_values(
        "total_revenue",
        ascending=False
    )
)

product_performance.head(10)

,product_id,product_name,total_revenue,total_units,total_orders,customers_served,branches_served,average_revenue_per_order
14,P015,Engine Block EB-600,5381418000,45222,3773,499,6,1.426297e+06
13,P014,Transmission Assembly TA-800,3760422500,45035,3838,500,6,9.797870e+05
20,P021,Boom Cylinder BC-400,2142016800,46364,3873,500,6,5.530640e+05
18,P019,Swing Motor SWM-350,1829645800,45971,3849,499,6,4.753561e+05
3,P004,Track Chain TC-45,1813885000,45575,3781,500,6,4.797368e+05
9,P010,Turbocharger TB-900,1639043600,46964,3933,500,6,4.167413e+05
10,P011,Hydraulic Cylinder HC-200,1286491200,46612,3982,500,6,3.230766e+05
0,P001,Hydraulic Pump HP-300,1112030500,45389,3854,500,6,2.885393e+05
29,P030,Fuel Pump FP-350,1005144300,45897,3856,500,6,2.606702e+05
21,P022,Control Valve CV-75,774341100,45819,3892,500,6,1.989571e+05


### **Product Performance Score**

In [7]:
product_performance["revenue_score"] = (
    product_performance["total_revenue"]
    / product_performance["total_revenue"].max()
    * 100
)

product_performance["unit_score"] = (
    product_performance["total_units"]
    / product_performance["total_units"].max()
    * 100
)

product_performance["order_score"] = (
    product_performance["total_orders"]
    / product_performance["total_orders"].max()
    * 100
)

product_performance["performance_score"] = (
    product_performance["revenue_score"] * 0.50
    + product_performance["unit_score"] * 0.30
    + product_performance["order_score"] * 0.20
)

### **Product Classification**

In [8]:
high_product_threshold = (
    product_performance["performance_score"]
    .quantile(0.75)
)

low_product_threshold = (
    product_performance["performance_score"]
    .quantile(0.25)
)

product_performance["performance_category"] = np.select(
    [
        product_performance["performance_score"]
        >= high_product_threshold,

        product_performance["performance_score"]
        <= low_product_threshold
    ],
    [
        "High Performance",
        "Low Performance"
    ],
    default="Medium Performance"
)

### **Warehouse/Branch Performance**

In [9]:
warehouse_performance = (
    week11_data
    .groupby("branch_id")
    .agg(
        total_orders=("so_id", "nunique"),
        total_units=("quantity", "sum"),
        total_revenue=("line_total", "sum"),
        products_handled=("product_id", "nunique"),
        customers_served=("customer_id", "nunique")
    )
    .reset_index()
)

warehouse_performance["average_order_value"] = (
    warehouse_performance["total_revenue"]
    / warehouse_performance["total_orders"]
)

### **Warehouse Performance Score**

In [10]:
warehouse_performance["revenue_score"] = (
    warehouse_performance["total_revenue"]
    / warehouse_performance["total_revenue"].max()
    * 100
)

warehouse_performance["order_score"] = (
    warehouse_performance["total_orders"]
    / warehouse_performance["total_orders"].max()
    * 100
)

warehouse_performance["unit_score"] = (
    warehouse_performance["total_units"]
    / warehouse_performance["total_units"].max()
    * 100
)

warehouse_performance["product_variety_score"] = (
    warehouse_performance["products_handled"]
    / warehouse_performance["products_handled"].max()
    * 100
)

warehouse_performance["warehouse_score"] = (
    warehouse_performance["revenue_score"] * 0.40
    + warehouse_performance["order_score"] * 0.25
    + warehouse_performance["unit_score"] * 0.25
    + warehouse_performance["product_variety_score"] * 0.10
)

### **Customer Performance**

In [11]:
customer_performance = (
    week11_data
    .groupby("customer_id")
    .agg(
        total_revenue=("line_total", "sum"),
        total_units=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        products_purchased=("product_id", "nunique"),
        branches_used=("branch_id", "nunique")
    )
    .reset_index()
)

customer_performance["average_order_value"] = (
    customer_performance["total_revenue"]
    / customer_performance["total_orders"]
)

### **Customer Segmentation**

In [12]:
high_value_threshold = (
    customer_performance["total_revenue"]
    .quantile(0.75)
)

low_value_threshold = (
    customer_performance["total_revenue"]
    .quantile(0.25)
)

customer_performance["customer_segment"] = np.select(
    [
        customer_performance["total_revenue"]
        >= high_value_threshold,

        customer_performance["total_revenue"]
        <= low_value_threshold
    ],
    [
        "High Value",
        "Low Value"
    ],
    default="Medium Value"
)

### **Repeat Customer Rate**

In [13]:
customer_performance["customer_status"] = np.where(
    customer_performance["total_orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

repeat_customer_rate = (
    customer_performance["customer_status"]
    .eq("Repeat Customer")
    .mean() * 100
)

print(
    f"Repeat Customer Rate: "
    f"{repeat_customer_rate:.2f}%"
)

Repeat Customer Rate: 100.00%


### **Inventory Requirements**

In [14]:
inventory_requirements = (
    week11_data
    .groupby(
        ["product_id", "product_name"],
        as_index=False
    )
    .agg(
        unit_cost=("unit_cost", "first"),
        safety_stock=("safety_stock", "first"),
        reorder_level=("reorder_level", "first"),
        max_stock_level=("max_stock_level", "first"),
        lead_time_days=("lead_time_days", "first"),
        units_sold=("quantity", "sum")
    )
)

inventory_requirements["safety_stock_value"] = (
    inventory_requirements["safety_stock"]
    * inventory_requirements["unit_cost"]
)

inventory_requirements["reorder_stock_value"] = (
    inventory_requirements["reorder_level"]
    * inventory_requirements["unit_cost"]
)

inventory_requirements["maximum_stock_value"] = (
    inventory_requirements["max_stock_level"]
    * inventory_requirements["unit_cost"]
)

### **Product-Warehouse Alignment**

In [15]:
product_warehouse = (
    week11_data
    .groupby(
        [
            "branch_id",
            "product_id",
            "product_name"
        ],
        as_index=False
    )
    .agg(
        total_units=("quantity", "sum"),
        total_revenue=("line_total", "sum"),
        total_orders=("so_id", "nunique"),
        customers=("customer_id", "nunique")
    )
)

product_warehouse["warehouse_revenue_share"] = (
    product_warehouse["total_revenue"]
    /
    product_warehouse.groupby("branch_id")
    ["total_revenue"]
    .transform("sum")
    * 100
)

### **Customer Revenue Concentration**

In [16]:
customer_concentration = (
    customer_performance[
        ["customer_id", "total_revenue"]
    ]
    .sort_values(
        "total_revenue",
        ascending=False
    )
    .reset_index(drop=True)
)

customer_concentration["revenue_share"] = (
    customer_concentration["total_revenue"]
    / customer_concentration["total_revenue"].sum()
    * 100
)

customer_concentration[
    "cumulative_revenue_share"
] = (
    customer_concentration["revenue_share"]
    .cumsum()
)

## **Forecast Summary**

In [18]:
forecast_data = pd.read_csv(
    "/content/future_sales_forecast.csv"
)

forecast_data.head()

,forecast_date,forecasted_revenue
0,2025-01-31,3.407833e+08
1,2025-02-28,3.215926e+08
2,2025-03-31,3.552769e+08
3,2025-04-30,3.452771e+08
4,2025-05-31,3.500754e+08


### **Final Integrated KPI Table**

In [19]:
week11_kpis = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Total Orders",
        "Total Units Sold",
        "Total Products",
        "Total Customers",
        "Total Branches",
        "Average Order Value",
        "Repeat Customer Rate",
        "Average Product Score",
        "Average Warehouse Score",
        "Safety Stock Value",
        "Reorder Stock Value",
        "Maximum Stock Value"
    ],
    "Value": [
        total_revenue,
        total_orders,
        total_units,
        total_products,
        total_customers,
        total_branches,
        average_order_value,
        repeat_customer_rate,
        product_performance["performance_score"].mean(),
        warehouse_performance["warehouse_score"].mean(),
        inventory_requirements["safety_stock_value"].sum(),
        inventory_requirements["reorder_stock_value"].sum(),
        inventory_requirements["maximum_stock_value"].sum()
    ]
})

week11_kpis

,KPI,Value
0,Total Revenue,2.544739e+10
1,Total Orders,2.000000e+04
2,Total Units Sold,1.368534e+06
3,Total Products,3.000000e+01
4,Total Customers,5.000000e+02
5,Total Branches,6.000000e+00
6,Average Order Value,1.272370e+06
7,Repeat Customer Rate,1.000000e+02
8,Average Product Score,5.633520e+01
9,Average Warehouse Score,8.349717e+01


### **Export Outputs for Python Dashboard**

In [20]:
powerbi_path = Path("/content/powerbi_data")

powerbi_path.mkdir(
    parents=True,
    exist_ok=True
)

In [21]:
week11_kpis.to_csv(
    powerbi_path / "week11_kpis.csv",
    index=False
)

monthly_sales.to_csv(
    powerbi_path / "monthly_sales.csv",
    index=False
)

product_performance.to_csv(
    powerbi_path / "product_performance.csv",
    index=False
)

warehouse_performance.to_csv(
    powerbi_path / "warehouse_performance.csv",
    index=False
)

customer_performance.to_csv(
    powerbi_path / "customer_performance.csv",
    index=False
)

inventory_requirements.to_csv(
    powerbi_path / "inventory_requirements.csv",
    index=False
)

product_warehouse.to_csv(
    powerbi_path / "product_warehouse.csv",
    index=False
)

customer_concentration.to_csv(
    powerbi_path / "customer_concentration.csv",
    index=False
)

## **Python Business Intelligence Dashboard**

The Week 11 dashboard will be built entirely in Python using **Plotly**.  
The notebook will produce six interactive dashboard pages as standalone HTML files:

1. Executive Overview
2. Product & Inventory Performance
3. Warehouse Performance
4. Customer Performance
5. Sales Forecast & Demand
6. Business Insights & Recommendations

The dashboard uses the Week 11 analytical dataframes already created above, rather than loading the large `sales_integrated.csv` again.

In [23]:
# Python Dashboard Setup

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

dashboard_path = Path("/content/week11_dashboard")
dashboard_path.mkdir(parents=True, exist_ok=True)

print("Dashboard output folder:", dashboard_path)

Dashboard output folder: /content/week11_dashboard


### **Additional Dashboard Data**

Some dashboard views require dimensions that are not present in the existing summary dataframes.  
These are calculated here from `week11_data` and kept as small aggregated tables for dashboard use.

In [24]:
# Additional aggregated tables for the Python dashboard

region_analysis = (
    week11_data
    .groupby("region", as_index=False)
    .agg(
        total_revenue=("line_total", "sum"),
        total_units=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        total_customers=("customer_id", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

customer_type_analysis = (
    week11_data
    .groupby("customer_type", as_index=False)
    .agg(
        total_revenue=("line_total", "sum"),
        total_units=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        total_customers=("customer_id", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

industry_segment_analysis = (
    week11_data
    .groupby("industry_segment", as_index=False)
    .agg(
        total_revenue=("line_total", "sum"),
        total_units=("quantity", "sum"),
        total_orders=("so_id", "nunique"),
        total_customers=("customer_id", "nunique")
    )
    .sort_values("total_revenue", ascending=False)
)

monthly_customer_activity = (
    week11_data
    .groupby(pd.Grouper(key="order_date", freq="ME"))
    .agg(
        active_customers=("customer_id", "nunique")
    )
    .reset_index()
)

product_demand = (
    week11_data
    .groupby(["product_id", "product_name"], as_index=False)
    .agg(total_units_sold=("quantity", "sum"))
    .sort_values("total_units_sold", ascending=False)
)

print("Additional dashboard tables created.")

Additional dashboard tables created.


## **Dashboard 1 — Executive Overview**

In [44]:
# ============================================================
# DASHBOARD 1 — EXECUTIVE OVERVIEW
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

# Top 10 products by revenue
top_10_products = (
    product_performance
    .nlargest(10, "total_revenue")
    .sort_values("total_revenue", ascending=True)
)

# Revenue by region
# Calculate directly from the integrated dataset to avoid
# depending on the column names in region_analysis.
region_chart = (
    week11_data
    .groupby("region", as_index=False)
    .agg(revenue=("line_total", "sum"))
    .sort_values("revenue", ascending=True)
)

# ------------------------------------------------------------
# Create dashboard layout
# ------------------------------------------------------------

fig1 = make_subplots(
    rows=2,
    cols=2,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    subplot_titles=[
        "Monthly Revenue Trend",
        "Top 10 Products by Revenue",
        "Revenue by Region",
        "Executive KPIs"
    ],
    horizontal_spacing=0.18,
    vertical_spacing=0.22,
    row_heights=[0.55, 0.45]
)

# ------------------------------------------------------------
# 1. Monthly Revenue Trend
# ------------------------------------------------------------

fig1.add_trace(
    go.Scatter(
        x=monthly_sales["order_date"],
        y=monthly_sales["revenue"],
        mode="lines+markers",
        name="Monthly Revenue",
        line=dict(width=2),
        marker=dict(size=6),
        hovertemplate=(
            "<b>%{x|%b %Y}</b><br>"
            "Revenue: £%{y:,.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# ------------------------------------------------------------
# 2. Top 10 Products by Revenue
# ------------------------------------------------------------

fig1.add_trace(
    go.Bar(
        x=top_10_products["total_revenue"],
        y=top_10_products["product_name"],
        orientation="h",
        name="Product Revenue",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Revenue: £%{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ------------------------------------------------------------
# 3. Revenue by Region
# ------------------------------------------------------------

fig1.add_trace(
    go.Bar(
        x=region_chart["revenue"],
        y=region_chart["region"],
        orientation="h",
        name="Regional Revenue",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Revenue: £%{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

# ------------------------------------------------------------
# 4. Executive KPI Panel
# ------------------------------------------------------------

# KPI values
kpi_values = [
    ("Average Order Value", f"£{average_order_value:,.2f}"),
    ("Repeat Customer Rate", f"{repeat_customer_rate:.1f}%"),
    ("Total Products", f"{total_products:,}"),
    ("Total Customers", f"{total_customers:,}"),
    ("Total Orders", f"{total_orders:,}")
]

# KPI card positions inside bottom-right area
kpi_positions = [
    (0.73, 0.30, "Average Order Value", f"£{average_order_value:,.2f}"),
    (0.88, 0.30, "Repeat Customer Rate", f"{repeat_customer_rate:.1f}%"),
    (0.73, 0.12, "Total Products", f"{total_products:,}"),
    (0.88, 0.12, "Total Customers", f"{total_customers:,}"),
    (0.805, 0.21, "Total Orders", f"{total_orders:,}")
]

for x, y, label, value in kpi_positions:

    fig1.add_annotation(
        x=x,
        y=y,
        xref="paper",
        yref="paper",
        text=(
            f"<b>{label}</b><br>"
            f"<span style='font-size:22px'>{value}</span>"
        ),
        showarrow=False,
        align="center",
        font=dict(
            size=14
        ),
        bgcolor="white",
        bordercolor="#D9E2F3",
        borderwidth=1,
        borderpad=12
    )

# ------------------------------------------------------------
# Dashboard formatting
# ------------------------------------------------------------

fig1.update_layout(
    title=dict(
        text="CadetX Week 11 — Executive Overview",
        x=0.5,
        font=dict(size=28)
    ),

    height=1000,
    width=1700,

    template="plotly_white",

    showlegend=False,

    margin=dict(
        l=130,
        r=100,
        t=130,
        b=100
    )
)

# ------------------------------------------------------------
# Dashboard formatting
# ------------------------------------------------------------

fig1.update_layout(
    title=dict(
        text="Executive Overview",
        x=0.5,
        font=dict(size=28)
    ),

    height=1000,
    width=1700,

    template="plotly_white",

    showlegend=False,

    margin=dict(
        l=130,
        r=130,
        t=130,
        b=100
    )
)

# ------------------------------------------------------------
# Axis formatting
# ------------------------------------------------------------

# Monthly revenue
fig1.update_xaxes(
    title_text="",
    automargin=True,
    row=1,
    col=1
)

fig1.update_yaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=1,
    col=1
)

# Product revenue
fig1.update_xaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=1,
    col=2
)

fig1.update_yaxes(
    title_text="",
    automargin=True,
    tickfont=dict(size=13),
    row=1,
    col=2
)

# Regional revenue
fig1.update_xaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=2,
    col=1
)

fig1.update_yaxes(
    title_text="",
    automargin=True,
    tickfont=dict(size=14),
    row=2,
    col=1
)

# ------------------------------------------------------------
# Export dashboard
# ------------------------------------------------------------

fig1.write_html(
    dashboard_path / "01_executive_overview.html",
    include_plotlyjs="cdn"
)

fig1.show()

## **Dashboard 2 — Product & Inventory Performance**

In [43]:
# ============================================================
# DASHBOARD 2 — PRODUCT & INVENTORY PERFORMANCE
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# Make sure dashboard folder exists
# ------------------------------------------------------------

dashboard_path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

# Top 10 products by revenue
top_10_revenue = (
    product_performance
    .nlargest(10, "total_revenue")
    .sort_values("total_revenue", ascending=True)
)

# Top 10 products by units sold
top_10_units = (
    product_performance
    .nlargest(10, "total_units")
    .sort_values("total_units", ascending=True)
)

# Top 10 products by maximum stock value
inventory_chart = (
    inventory_requirements
    .nlargest(10, "maximum_stock_value")
    .sort_values("maximum_stock_value", ascending=True)
)

# ------------------------------------------------------------
# Create dashboard
# ------------------------------------------------------------

fig2 = make_subplots(
    rows=2,
    cols=2,

    subplot_titles=[
        "Top 10 Products by Revenue",
        "Top 10 Products by Units Sold",
        "Inventory Requirement Values",
        "Inventory KPIs"
    ],

    horizontal_spacing=0.18,
    vertical_spacing=0.25,

    row_heights=[0.52, 0.48]
)

# ------------------------------------------------------------
# 1. Top 10 Products by Revenue
# ------------------------------------------------------------

fig2.add_trace(
    go.Bar(
        x=top_10_revenue["total_revenue"],
        y=top_10_revenue["product_name"],
        orientation="h",
        name="Revenue",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Revenue: £%{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# ------------------------------------------------------------
# 2. Top 10 Products by Units Sold
# ------------------------------------------------------------

fig2.add_trace(
    go.Bar(
        x=top_10_units["total_units"],
        y=top_10_units["product_name"],
        orientation="h",
        name="Units Sold",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Units Sold: %{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ------------------------------------------------------------
# 3. Inventory Requirement Values
# ------------------------------------------------------------

fig2.add_trace(
    go.Bar(
        x=inventory_chart["product_name"],
        y=inventory_chart["safety_stock_value"],
        name="Safety Stock",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Safety Stock: £%{y:,.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

fig2.add_trace(
    go.Bar(
        x=inventory_chart["product_name"],
        y=inventory_chart["reorder_stock_value"],
        name="Reorder Stock",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Reorder Stock: £%{y:,.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

fig2.add_trace(
    go.Bar(
        x=inventory_chart["product_name"],
        y=inventory_chart["maximum_stock_value"],
        name="Maximum Stock",
        hovertemplate=(
            "<b>%{x}</b><br>"
            "Maximum Stock: £%{y:,.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

# ------------------------------------------------------------
# 4. Inventory KPI Panel
# ------------------------------------------------------------

safety_value = inventory_requirements["safety_stock_value"].sum()
reorder_value = inventory_requirements["reorder_stock_value"].sum()
maximum_value = inventory_requirements["maximum_stock_value"].sum()

# KPI panel background
fig2.add_shape(
    type="rect",
    x0=0,
    y0=0,
    x1=1,
    y1=1,
    xref="x4",
    yref="y4",
    line=dict(
        color="lightgray",
        width=1
    ),
    fillcolor="white",
    layer="below",
    row=2,
    col=2
)

# KPI text
kpi_text = (
    f"<b>Total Products</b><br>"
    f"<span style='font-size:22px'>{total_products:,}</span>"
    f"<br><br>"

    f"<b>Total Units Sold</b><br>"
    f"<span style='font-size:22px'>{total_units:,}</span>"
    f"<br><br>"

    f"<b>Safety Stock Value</b><br>"
    f"<span style='font-size:20px'>£{safety_value:,.0f}</span>"
    f"<br><br>"

    f"<b>Reorder Stock Value</b><br>"
    f"<span style='font-size:20px'>£{reorder_value:,.0f}</span>"
    f"<br><br>"

    f"<b>Maximum Stock Value</b><br>"
    f"<span style='font-size:20px'>£{maximum_value:,.0f}</span>"
)

fig2.add_annotation(
    x=0.5,
    y=0.5,
    xref="x4",
    yref="y4",
    text=kpi_text,
    showarrow=False,
    align="center",
    font=dict(size=14),
    xanchor="center",
    yanchor="middle",
    row=2,
    col=2
)

# Hide the KPI subplot axes
fig2.update_xaxes(
    range=[0, 1],
    visible=False,
    row=2,
    col=2
)

fig2.update_yaxes(
    range=[0, 1],
    visible=False,
    row=2,
    col=2
)

# ------------------------------------------------------------
# Layout
# ------------------------------------------------------------

fig2.update_layout(
    title=dict(
        text="Product & Inventory Performance",
        x=0.5,
        font=dict(size=28)
    ),

    height=1050,
    width=1800,

    template="plotly_white",

    barmode="group",

    showlegend=True,

    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.12,
        xanchor="center",
        x=0.5
    ),

    margin=dict(
        l=150,
        r=100,
        t=130,
        b=130
    )
)

# ------------------------------------------------------------
# Axis formatting
# ------------------------------------------------------------

# Revenue chart
fig2.update_xaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=1,
    col=1
)

fig2.update_yaxes(
    automargin=True,
    tickfont=dict(size=13),
    row=1,
    col=1
)

# Units chart
fig2.update_xaxes(
    title_text="Units Sold",
    tickformat="~s",
    automargin=True,
    row=1,
    col=2
)

fig2.update_yaxes(
    automargin=True,
    tickfont=dict(size=13),
    row=1,
    col=2
)

# Inventory chart
fig2.update_yaxes(
    title_text="Stock Value",
    tickformat="~s",
    automargin=True,
    row=2,
    col=1
)

fig2.update_xaxes(
    tickangle=-35,
    automargin=True,
    row=2,
    col=1
)

# ------------------------------------------------------------
# Export HTML
# ------------------------------------------------------------

fig2.write_html(
    dashboard_path / "02_product_inventory.html",
    include_plotlyjs="cdn"
)

# ------------------------------------------------------------
# Display dashboard
# ------------------------------------------------------------

fig2.show()

## **Dashboard 3 — Warehouse Performance**

In [42]:
# ============================================================
# DASHBOARD 3 — WAREHOUSE PERFORMANCE
# ============================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# Prepare data
# ------------------------------------------------------------

dashboard_path.mkdir(parents=True, exist_ok=True)

# Warehouse performance
warehouse_chart = (
    warehouse_performance
    .sort_values("total_revenue", ascending=True)
)

# Top product–warehouse combinations
product_warehouse_chart = (
    product_warehouse
    .copy()
)

product_warehouse_chart["product_warehouse"] = (
    product_warehouse_chart["product_name"]
    + " — "
    + product_warehouse_chart["branch_id"].astype(str)
)

product_warehouse_top = (
    product_warehouse_chart
    .nlargest(12, "total_revenue")
    .sort_values("total_revenue", ascending=True)
)

# ------------------------------------------------------------
# Create dashboard
# ------------------------------------------------------------

fig3 = make_subplots(
    rows=2,
    cols=2,

    subplot_titles=[
        "Orders vs Revenue",
        "Top Product–Warehouse Revenue Combinations",
        "Revenue by Warehouse",
        "Warehouse Performance Score"
    ],

    horizontal_spacing=0.18,
    vertical_spacing=0.25,

    row_heights=[0.52, 0.48]
)

# ============================================================
# 1. Orders vs Revenue
# ============================================================

fig3.add_trace(
    go.Scatter(
        x=warehouse_performance["total_orders"],
        y=warehouse_performance["total_revenue"],
        mode="markers",

        marker=dict(
            size=18,
            opacity=0.75
        ),

        name="Warehouse",

        customdata=warehouse_performance[
            [
                "branch_id",
                "total_orders",
                "total_units",
                "total_revenue",
                "warehouse_score"
            ]
        ],

        hovertemplate=(
            "<b>Branch: %{customdata[0]}</b><br>"
            "Orders: %{customdata[1]:,.0f}<br>"
            "Units: %{customdata[2]:,.0f}<br>"
            "Revenue: £%{customdata[3]:,.0f}<br>"
            "Warehouse Score: %{customdata[4]:.1f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# ============================================================
# 2. Top Product–Warehouse Revenue Combinations
# ============================================================

fig3.add_trace(
    go.Bar(
        x=product_warehouse_top["total_revenue"],
        y=product_warehouse_top["product_warehouse"],
        orientation="h",

        name="Product–Warehouse Revenue",

        hovertemplate=(
            "<b>%{y}</b><br>"
            "Revenue: £%{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# ============================================================
# 3. Revenue by Warehouse
# ============================================================

fig3.add_trace(
    go.Bar(
        x=warehouse_chart["total_revenue"],
        y=warehouse_chart["branch_id"].astype(str),
        orientation="h",

        name="Warehouse Revenue",

        hovertemplate=(
            "<b>Branch: %{y}</b><br>"
            "Revenue: £%{x:,.0f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=1
)

# ============================================================
# 4. Warehouse Performance Score
# ============================================================

warehouse_score_chart = (
    warehouse_performance
    .sort_values("warehouse_score", ascending=True)
)

fig3.add_trace(
    go.Bar(
        x=warehouse_score_chart["warehouse_score"],
        y=warehouse_score_chart["branch_id"].astype(str),
        orientation="h",

        name="Warehouse Score",

        hovertemplate=(
            "<b>Branch: %{y}</b><br>"
            "Performance Score: %{x:.1f}"
            "<extra></extra>"
        )
    ),
    row=2,
    col=2
)

# ============================================================
# Layout
# ============================================================

fig3.update_layout(

    title=dict(
        text="Warehouse Performance",
        x=0.5,
        font=dict(size=28)
    ),

    height=1050,
    width=1800,

    template="plotly_white",

    showlegend=False,

    margin=dict(
        l=150,
        r=120,
        t=130,
        b=130
    )
)

# ============================================================
# Axis formatting
# ============================================================

# Orders vs Revenue
fig3.update_xaxes(
    title_text="Total Orders",
    tickformat="~s",
    automargin=True,
    row=1,
    col=1
)

fig3.update_yaxes(
    title_text="Total Revenue",
    tickformat="~s",
    automargin=True,
    row=1,
    col=1
)

# Product–Warehouse combinations
fig3.update_xaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=1,
    col=2
)

fig3.update_yaxes(
    title_text="",
    tickfont=dict(size=12),
    automargin=True,
    row=1,
    col=2
)

# Revenue by warehouse
fig3.update_xaxes(
    title_text="Revenue",
    tickformat="~s",
    automargin=True,
    row=2,
    col=1
)

fig3.update_yaxes(
    title_text="Branch",
    tickfont=dict(size=13),
    automargin=True,
    row=2,
    col=1
)

# Warehouse score
fig3.update_xaxes(
    title_text="Performance Score",
    automargin=True,
    row=2,
    col=2
)

fig3.update_yaxes(
    title_text="Branch",
    tickfont=dict(size=13),
    automargin=True,
    row=2,
    col=2
)

# ============================================================
# Export
# ============================================================

fig3.write_html(
    dashboard_path / "03_warehouse_performance.html",
    include_plotlyjs="cdn"
)

# ============================================================
# Display
# ============================================================

fig3.show()

## **Dashboard 4 — Customer Performance**

In [41]:
# Dashboard 4: Customer Performance

top10_customers = customer_performance.nlargest(10, "total_revenue").sort_values("total_revenue")
segment_summary = (
    customer_performance
    .groupby("customer_segment", as_index=False)
    .agg(
        customers=("customer_id", "nunique"),
        revenue=("total_revenue", "sum")
    )
)

fig = make_subplots(
    rows=3, cols=2,
    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "xy"}, {"type": "xy"}]
    ],
    subplot_titles=(
        "Top 10 Customers by Revenue",
        "Customer Segments",
        "Revenue by Customer Type",
        "Revenue by Industry Segment",
        "Monthly Active Customers",
        "Customer Revenue Concentration"
    ),
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

fig.add_trace(go.Bar(
    x=top10_customers["total_revenue"],
    y=top10_customers["customer_id"].astype(str),
    orientation="h",
    name="Revenue"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=segment_summary["customers"],
    y=segment_summary["customer_segment"],
    orientation="h",
    name="Customers"
), row=1, col=2)

fig.add_trace(go.Bar(
    x=customer_type_analysis["total_revenue"],
    y=customer_type_analysis["customer_type"].astype(str),
    orientation="h",
    name="Revenue"
), row=2, col=1)

industry_top = industry_segment_analysis.head(10).sort_values("total_revenue")
fig.add_trace(go.Bar(
    x=industry_top["total_revenue"],
    y=industry_top["industry_segment"].astype(str),
    orientation="h",
    name="Revenue"
), row=2, col=2)

fig.add_trace(go.Scatter(
    x=monthly_customer_activity["order_date"],
    y=monthly_customer_activity["active_customers"],
    mode="lines+markers",
    name="Active Customers"
), row=3, col=1)

rank = list(range(1, len(customer_concentration) + 1))
fig.add_trace(go.Scatter(
    x=rank,
    y=customer_concentration["cumulative_revenue_share"],
    mode="lines",
    name="Cumulative Revenue Share"
), row=3, col=2)

fig.update_layout(
    title="Customer Performance",
    height=1200,
    showlegend=False,
    template="plotly_white"
)

fig.update_yaxes(title_text="Revenue", row=3, col=2)
fig.update_xaxes(title_text="Customer Rank", row=3, col=2)

fig.show()
fig.write_html(dashboard_path / "04_customer_performance.html", include_plotlyjs=True)

## **Dashboard 5 — Sales Forecast & Demand**

In [40]:
# ============================================================
# DASHBOARD 5 — SALES FORECAST & DEMAND
# ============================================================

from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ------------------------------------------------------------
# Dashboard output folder
# ------------------------------------------------------------

dashboard_path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load forecast data
# ------------------------------------------------------------

forecast_file = Path("/content/future_sales_forecast.csv")

if forecast_file.exists():

    forecast_data = pd.read_csv(forecast_file)

    print("Forecast columns:", forecast_data.columns.tolist())

    # --------------------------------------------------------
    # Detect the actual columns in the Week 9 forecast file
    # --------------------------------------------------------

    date_candidates = [
        "forecast_date",
        "order_date",
        "date",
        "ds"
    ]

    revenue_candidates = [
        "forecasted_revenue",   # Actual column in your file
        "forecast_revenue",
        "forecast",
        "predicted_revenue",
        "yhat",
        "revenue"
    ]

    forecast_date_col = next(
        (
            col for col in date_candidates
            if col in forecast_data.columns
        ),
        None
    )

    forecast_value_col = next(
        (
            col for col in revenue_candidates
            if col in forecast_data.columns
        ),
        None
    )

    # --------------------------------------------------------
    # Check forecast columns
    # --------------------------------------------------------

    if forecast_date_col is None or forecast_value_col is None:

        print(
            "Forecast file found, but the required "
            "date/revenue columns were not detected."
        )

        print(
            "Available columns:",
            forecast_data.columns.tolist()
        )

    else:

        print(
            "Using forecast date column:",
            forecast_date_col
        )

        print(
            "Using forecast revenue column:",
            forecast_value_col
        )

        # ----------------------------------------------------
        # Clean forecast data
        # ----------------------------------------------------

        forecast_data[forecast_date_col] = pd.to_datetime(
            forecast_data[forecast_date_col],
            errors="coerce"
        )

        forecast_data[forecast_value_col] = pd.to_numeric(
            forecast_data[forecast_value_col],
            errors="coerce"
        )

        forecast_data = (
            forecast_data
            .dropna(
                subset=[
                    forecast_date_col,
                    forecast_value_col
                ]
            )
            .sort_values(forecast_date_col)
        )

        # ----------------------------------------------------
        # Prepare top product demand
        # ----------------------------------------------------

        if "product_demand" in globals():

            top_demand = (
                product_demand
                .nlargest(10, "total_units_sold")
                .sort_values(
                    "total_units_sold",
                    ascending=True
                )
            )

        else:

            product_demand = (
                week11_data
                .groupby(
                    ["product_id", "product_name"],
                    as_index=False
                )
                .agg(
                    total_units_sold=(
                        "quantity",
                        "sum"
                    )
                )
            )

            top_demand = (
                product_demand
                .nlargest(10, "total_units_sold")
                .sort_values(
                    "total_units_sold",
                    ascending=True
                )
            )

        # ----------------------------------------------------
        # Prepare monthly sales data
        # ----------------------------------------------------

        monthly_sales_dashboard = monthly_sales.copy()

        monthly_sales_dashboard["order_date"] = pd.to_datetime(
            monthly_sales_dashboard["order_date"],
            errors="coerce"
        )

        monthly_sales_dashboard = (
            monthly_sales_dashboard
            .dropna(subset=["order_date"])
            .sort_values("order_date")
        )

        # ----------------------------------------------------
        # Create dashboard
        # ----------------------------------------------------

        fig5 = make_subplots(
            rows=2,
            cols=2,

            subplot_titles=[
                "Historical Revenue",
                "Forecast Revenue",
                "Monthly Revenue Growth",
                "Top 10 Product Demand"
            ],

            horizontal_spacing=0.18,
            vertical_spacing=0.24,

            row_heights=[0.52, 0.48]
        )

        # ====================================================
        # 1. Historical Revenue
        # ====================================================

        fig5.add_trace(
            go.Scatter(
                x=monthly_sales_dashboard["order_date"],
                y=monthly_sales_dashboard["revenue"],
                mode="lines+markers",
                name="Historical Revenue",

                line=dict(width=2),
                marker=dict(size=5),

                hovertemplate=(
                    "<b>%{x|%b %Y}</b><br>"
                    "Revenue: £%{y:,.0f}"
                    "<extra></extra>"
                )
            ),
            row=1,
            col=1
        )

        # ====================================================
        # 2. Forecast Revenue
        # ====================================================

        fig5.add_trace(
            go.Scatter(
                x=forecast_data[forecast_date_col],
                y=forecast_data[forecast_value_col],
                mode="lines+markers",
                name="Forecast Revenue",

                line=dict(width=2),
                marker=dict(size=6),

                hovertemplate=(
                    "<b>%{x|%b %Y}</b><br>"
                    "Forecast Revenue: £%{y:,.0f}"
                    "<extra></extra>"
                )
            ),
            row=1,
            col=2
        )

        # ====================================================
        # 3. Monthly Revenue Growth
        # ====================================================

        fig5.add_trace(
            go.Scatter(
                x=monthly_sales_dashboard["order_date"],
                y=monthly_sales_dashboard["revenue_growth"],
                mode="lines+markers",
                name="Revenue Growth",

                line=dict(width=2),
                marker=dict(size=5),

                hovertemplate=(
                    "<b>%{x|%b %Y}</b><br>"
                    "Revenue Growth: %{y:.1f}%"
                    "<extra></extra>"
                )
            ),
            row=2,
            col=1
        )

        # ====================================================
        # 4. Top Product Demand
        # ====================================================

        fig5.add_trace(
            go.Bar(
                x=top_demand["total_units_sold"],
                y=top_demand["product_name"],
                orientation="h",
                name="Units Sold",

                hovertemplate=(
                    "<b>%{y}</b><br>"
                    "Units Sold: %{x:,.0f}"
                    "<extra></extra>"
                )
            ),
            row=2,
            col=2
        )

        # ====================================================
        # Dashboard Layout
        # ====================================================

        fig5.update_layout(

            title=dict(
                text="Sales Forecast & Demand",
                x=0.5,
                font=dict(size=28)
            ),

            height=1050,
            width=1800,

            template="plotly_white",

            showlegend=False,

            margin=dict(
                l=150,
                r=120,
                t=130,
                b=130
            )
        )

        # ====================================================
        # Axis Formatting
        # ====================================================

        # Historical Revenue
        fig5.update_xaxes(
            title_text="Date",
            automargin=True,
            row=1,
            col=1
        )

        fig5.update_yaxes(
            title_text="Revenue",
            tickformat="~s",
            automargin=True,
            row=1,
            col=1
        )

        # Forecast Revenue
        fig5.update_xaxes(
            title_text="Forecast Date",
            automargin=True,
            row=1,
            col=2
        )

        fig5.update_yaxes(
            title_text="Forecast Revenue",
            tickformat="~s",
            automargin=True,
            row=1,
            col=2
        )

        # Revenue Growth
        fig5.update_xaxes(
            title_text="Date",
            automargin=True,
            row=2,
            col=1
        )

        fig5.update_yaxes(
            title_text="Growth (%)",
            ticksuffix="%",
            automargin=True,
            row=2,
            col=1
        )

        # Product Demand
        fig5.update_xaxes(
            title_text="Units Sold",
            tickformat="~s",
            automargin=True,
            row=2,
            col=2
        )

        fig5.update_yaxes(
            title_text="",
            tickfont=dict(size=12),
            automargin=True,
            row=2,
            col=2
        )

        # ----------------------------------------------------
        # Export dashboard
        # ----------------------------------------------------

        fig5.write_html(
            dashboard_path / "05_sales_forecast_demand.html",
            include_plotlyjs="cdn"
        )

        # ----------------------------------------------------
        # Display dashboard
        # ----------------------------------------------------

        fig5.show()

else:

    print(
        "Forecast file not found at "
        "/content/future_sales_forecast.csv."
    )

    print(
        "Run the Week 9 forecasting notebook first, "
        "then rerun this Dashboard 5 cell."
    )

Forecast columns: ['forecast_date', 'forecasted_revenue']
Using forecast date column: forecast_date
Using forecast revenue column: forecasted_revenue


## **Dashboard 6 — Business Insights & Recommendations**

This page is intentionally text-based. The statements should be populated from the actual results above rather than invented in advance.

In [46]:
# Dashboard 6: Business Insights & Recommendations

peak_month = monthly_sales.loc[monthly_sales["revenue"].idxmax(), "order_date"]
peak_revenue = monthly_sales["revenue"].max()

lowest_month = monthly_sales.loc[monthly_sales["revenue"].idxmin(), "order_date"]
lowest_revenue = monthly_sales["revenue"].min()

top_product = product_performance.iloc[0]["product_name"]
top_product_revenue = product_performance.iloc[0]["total_revenue"]

top_customer = customer_performance.nlargest(1, "total_revenue").iloc[0]

top_branch = warehouse_performance.nlargest(1, "total_revenue").iloc[0]

insights = [
    f"Sales: The highest monthly revenue was £{peak_revenue:,.0f} in {peak_month:%B %Y}.",
    f"Sales: The lowest monthly revenue was £{lowest_revenue:,.0f} in {lowest_month:%B %Y}.",
    f"Products: {top_product} generated the highest total revenue at £{top_product_revenue:,.0f}.",
    f"Customers: The repeat customer rate was {repeat_customer_rate:.1f}%.",
    f"Warehouses: Branch {top_branch['branch_id']} generated the highest branch revenue at £{top_branch['total_revenue']:,.0f}.",
    "Inventory: Safety stock, reorder-level and maximum-stock values represent calculated inventory requirements, not confirmed current inventory.",
]

print("=" * 70)
print("BUSINESS INSIGHTS & RECOMMENDATIONS")
print("=" * 70)

for item in insights:
    print("•", item)

print("\nRecommended management focus:")
print("• Monitor high-revenue and high-demand products.")
print("• Review branches with unusual workload/revenue patterns.")
print("• Use customer segmentation to identify retention and growth opportunities.")
print("• Use inventory requirement values to support procurement planning.")
print("• Use forecast results as an additional input to future demand planning.")

BUSINESS INSIGHTS & RECOMMENDATIONS
• Sales: The highest monthly revenue was £429,255,230 in July 2020.
• Sales: The lowest monthly revenue was £281,974,060 in February 2019.
• Products: Engine Block EB-600 generated the highest total revenue at £5,381,418,000.
• Customers: The repeat customer rate was 100.0%.
• Warehouses: Branch CHN001 generated the highest branch revenue at £5,192,125,100.
• Inventory: Safety stock, reorder-level and maximum-stock values represent calculated inventory requirements, not confirmed current inventory.

Recommended management focus:
• Monitor high-revenue and high-demand products.
• Review branches with unusual workload/revenue patterns.
• Use customer segmentation to identify retention and growth opportunities.
• Use inventory requirement values to support procurement planning.
• Use forecast results as an additional input to future demand planning.


## **Export Dashboard Data**

The following files are small analytical outputs for the Python dashboard. They can also be used for GitHub documentation or future BI tools.

In [47]:
# Export small Week 11 dashboard datasets

analysis_path = dashboard_path / "data"
analysis_path.mkdir(parents=True, exist_ok=True)

dashboard_exports = {
    "week11_kpis.csv": week11_kpis,
    "monthly_sales.csv": monthly_sales,
    "product_performance.csv": product_performance,
    "warehouse_performance.csv": warehouse_performance,
    "customer_performance.csv": customer_performance,
    "inventory_requirements.csv": inventory_requirements,
    "product_warehouse.csv": product_warehouse,
    "customer_concentration.csv": customer_concentration,
    "region_analysis.csv": region_analysis,
    "customer_type_analysis.csv": customer_type_analysis,
    "industry_segment_analysis.csv": industry_segment_analysis,
    "monthly_customer_activity.csv": monthly_customer_activity,
    "product_demand.csv": product_demand,
}

for filename, dataframe in dashboard_exports.items():
    dataframe.to_csv(analysis_path / filename, index=False)

print(f"Exported {len(dashboard_exports)} dashboard datasets to:")
print(analysis_path)

Exported 13 dashboard datasets to:
/content/week11_dashboard/data


## **Final Python Dashboard Outputs**

After running the cells above, the dashboard folder will contain:

```text
week11_dashboard/
├── 01_executive_overview.html
├── 02_product_inventory.html
├── 03_warehouse_performance.html
├── 04_customer_performance.html
├── 05_sales_forecast_demand.html
└── data/
    ├── week11_kpis.csv
    ├── monthly_sales.csv
    ├── product_performance.csv
    ├── warehouse_performance.csv
    ├── customer_performance.csv
    ├── inventory_requirements.csv
    ├── product_warehouse.csv
    ├── customer_concentration.csv
    ├── region_analysis.csv
    ├── customer_type_analysis.csv
    ├── industry_segment_analysis.csv
    ├── monthly_customer_activity.csv
    └── product_demand.csv
```

The sixth dashboard page is represented by the notebook's Business Insights section and can be converted to HTML/PDF later if required.